In [34]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM
import scipy

In [108]:
combined_df = pd.read_excel('Combined_Time_Fix_Data.xlsx')
Dataset_Values = combined_df['Dataset'].unique()
columns = combined_df.columns.values
print(combined_df[columns[3]].unique())
combined_df.drop(columns=columns[0], axis=1, inplace=True) # inplace to modify combined_df instead of copy of object
combined_df.info()
# combined_df[combined_df['Dataset']==Dataset_Values[2]].head()

['Chemotaxis_Diffusion' 'Conjugation_Ra' 'SOS_Spontaneous']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    800 non-null    object 
 1   Time         800 non-null    int64  
 2   Dataset      800 non-null    object 
 3   Log_Time     0 non-null      float64
 4   Conjugation  800 non-null    bool   
 5   R_a          800 non-null    bool   
 6   Chemotaxis   800 non-null    bool   
 7   SOS          800 non-null    bool   
dtypes: bool(4), float64(1), int64(1), object(2)
memory usage: 28.3+ KB


### Data Cleaning

In [104]:
# Removing Outliers
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Function to clean data per dataset
def clean_data_by_dataset(df, dataset_column, target_column):
    cleaned_df_list = []
    unique_datasets = df[dataset_column].unique() # Get unique names from 'Dataset' column.
    
    for dataset in unique_datasets:
        dataset_df = df[df[dataset_column] == dataset]
        cleaned_dataset_df = remove_outliers(dataset_df, target_column) # 'Time' is the target column to remove the outliers from.
        cleaned_df_list.append(cleaned_dataset_df)
    
    cleaned_combined_df = pd.concat(cleaned_df_list)
    cleaned_combined_df.reset_index(drop=True, inplace=True)
    return cleaned_combined_df

In [109]:
# Apply the function
combined_df = clean_data_by_dataset(combined_df, 'Condition', 'Time')

# Check the result
combined_df.info()
combined_df['Log_Time'] = combined_df['Time'].apply(lambda x: np.log(x))
combined_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 772 entries, 0 to 771
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    772 non-null    object 
 1   Time         772 non-null    int64  
 2   Dataset      772 non-null    object 
 3   Log_Time     0 non-null      float64
 4   Conjugation  772 non-null    bool   
 5   R_a          772 non-null    bool   
 6   Chemotaxis   772 non-null    bool   
 7   SOS          772 non-null    bool   
dtypes: bool(4), float64(1), int64(1), object(2)
memory usage: 27.3+ KB


,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS
0,Chemotaxis,16634,Chemotaxis_Diffusion,9.719204,True,True,True,True
1,Chemotaxis,15586,Chemotaxis_Diffusion,9.654128,True,True,True,True
2,Chemotaxis,15735,Chemotaxis_Diffusion,9.663643,True,True,True,True
3,Chemotaxis,15363,Chemotaxis_Diffusion,9.639717,True,True,True,True
4,Chemotaxis,16444,Chemotaxis_Diffusion,9.707716,True,True,True,True
...,...,...,...,...,...,...,...,...
767,Spontaneous,17196,SOS_Spontaneous,9.752432,True,True,True,False
768,Spontaneous,46726,SOS_Spontaneous,10.752056,True,True,True,False
769,Spontaneous,43138,SOS_Spontaneous,10.672160,True,True,True,False
770,Spontaneous,26322,SOS_Spontaneous,10.178160,True,True,True,False


### Conjugation R_a Regression Analysis

In [117]:
# Filter the dataframe to exclude the 'Chemotaxis_Diffusion' dataset
filtered_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()

# Create the new column with opposite boolean values
filtered_df["No_R_a"] = ~filtered_df["R_a"]

# Add interaction term for R_a * Conjugation
filtered_df['R_a_Conj'] = filtered_df['R_a'] & filtered_df['Conjugation']

# Add interaction term for No_R_a * Conjugation
filtered_df['No_R_a_Conj'] = filtered_df['No_R_a'] & filtered_df['Conjugation']

# Display the first few rows to verify the new column
filtered_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Conjugation', 'R_a', 'R_a_Conj', 'No_R_a_Conj', 'No_R_a']].head()

,Condition,Time,Dataset,Log_Time,Conjugation,R_a,R_a_Conj,No_R_a_Conj,No_R_a
187,Normal Model,3124,Conjugation_Ra,8.046870,True,True,True,False,False
188,Normal Model,6104,Conjugation_Ra,8.716700,True,True,True,False,False
189,Normal Model,3418,Conjugation_Ra,8.136811,True,True,True,False,False
190,Normal Model,4620,Conjugation_Ra,8.438150,True,True,True,False,False
191,Normal Model,4282,Conjugation_Ra,8.362175,True,True,True,False,False


In [118]:
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation + No_R_a'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.123
Model:                            OLS   Adj. R-squared:                  0.118
Method:                 Least Squares   F-statistic:                     27.00
Date:                Tue, 23 Jul 2024   Prob (F-statistic):           1.06e-11
Time:                        00:14:19   Log-Likelihood:                -193.34
No. Observations:                 389   AIC:                             392.7
Df Residuals:                     386   BIC:                             404.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               8.5348    

In [119]:
filtered_df['No_R_a_Conj'].info()
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation * No_R_a'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

<class 'pandas.core.series.Series'>
Index: 389 entries, 187 to 575
Series name: No_R_a_Conj
Non-Null Count  Dtype
--------------  -----
389 non-null    bool 
dtypes: bool(1)
memory usage: 3.4 KB
                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.140
Model:                            OLS   Adj. R-squared:                  0.133
Method:                 Least Squares   F-statistic:                     20.85
Date:                Tue, 23 Jul 2024   Prob (F-statistic):           1.54e-12
Time:                        00:14:27   Log-Likelihood:                -189.52
No. Observations:                 389   AIC:                             387.0
Df Residuals:                     385   BIC:                             402.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
               

In [120]:
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation + No_R_a_Conj'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.140
Model:                            OLS   Adj. R-squared:                  0.135
Method:                 Least Squares   F-statistic:                     31.34
Date:                Tue, 23 Jul 2024   Prob (F-statistic):           2.44e-13
Time:                        00:15:02   Log-Likelihood:                -189.53
No. Observations:                 389   AIC:                             385.1
Df Residuals:                     386   BIC:                             397.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               8.4829    

The interaction term R_a_Conj is statistically significant in the simpler model because this model directly captures the combined effect without the added degrees of freedom, larger standard errors, and potential collinearity of separate terms, thereby reducing the t-values and making it harder to achieve statistical significance.

**Conclusion of Analysis and Comparison to Permutation Test:**
- Conjugation combined with a lack of R_a is the most statistically significant predictor, coef = -0.3226 (p < 0.0001).
- R_a being absent is not significant.
- Conj being present is significant (p < 0.001).
- The permutation test showed practical significance for the interaction effect between the synergystic effect of no R(a) and conjugation.
- Regression results for No_R_a_Conj indicate a significant interaction effect in the model focusing on this term, although the regression model with both conjugation and no R(a) has a larger R-squared indicating a better fit with variance.

### Chemotaxis Diffusion Regression Analysis

In [121]:
# Filter the dataframe to include only the 'Chemotaxis_Diffusion' dataset
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()

# Display the first few rows to verify
chemotaxis_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Chemotaxis']]

,Condition,Time,Dataset,Log_Time,Chemotaxis
0,Chemotaxis,16634,Chemotaxis_Diffusion,9.719204,True
1,Chemotaxis,15586,Chemotaxis_Diffusion,9.654128,True
2,Chemotaxis,15735,Chemotaxis_Diffusion,9.663643,True
3,Chemotaxis,15363,Chemotaxis_Diffusion,9.639717,True
4,Chemotaxis,16444,Chemotaxis_Diffusion,9.707716,True
...,...,...,...,...,...
182,Diffusion,19385,Chemotaxis_Diffusion,9.872255,False
183,Diffusion,20579,Chemotaxis_Diffusion,9.932026,False
184,Diffusion,18450,Chemotaxis_Diffusion,9.822820,False
185,Diffusion,19071,Chemotaxis_Diffusion,9.855924,False


In [122]:
# Prepare the model formula
formula = 'Log_Time ~ Chemotaxis'

# Fit the OLS regression model
model_chemotaxis = sm.OLS.from_formula(formula, data=chemotaxis_df).fit()

# Display the summary of the model
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.566
Model:                            OLS   Adj. R-squared:                  0.563
Method:                 Least Squares   F-statistic:                     240.8
Date:                Tue, 23 Jul 2024   Prob (F-statistic):           2.50e-35
Time:                        00:15:59   Log-Likelihood:                 216.50
No. Observations:                 187   AIC:                            -429.0
Df Residuals:                     185   BIC:                            -422.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              9.9000      0

### Estimating Fitted Mean

In [110]:
sos_spontaneous_df_cleaned = combined_df[combined_df['Dataset'] == 'SOS_Spontaneous'].copy()
sos_spontaneous_df_cleaned

,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS
576,SOS,17623,SOS_Spontaneous,9.776960,True,True,True,True
577,SOS,21652,SOS_Spontaneous,9.982853,True,True,True,True
578,SOS,20515,SOS_Spontaneous,9.928912,True,True,True,True
579,SOS,19787,SOS_Spontaneous,9.892780,True,True,True,True
580,SOS,19868,SOS_Spontaneous,9.896866,True,True,True,True
...,...,...,...,...,...,...,...,...
767,Spontaneous,17196,SOS_Spontaneous,9.752432,True,True,True,False
768,Spontaneous,46726,SOS_Spontaneous,10.752056,True,True,True,False
769,Spontaneous,43138,SOS_Spontaneous,10.672160,True,True,True,False
770,Spontaneous,26322,SOS_Spontaneous,10.178160,True,True,True,False


In [111]:
# Replace censored times with the estimated mean time from Weibull distribution
def estimate_weibull_mean(params):
    shape, loc, scale = params
    return np.log(scale * scipy.special.gamma(1 + 1/shape))

# Filter data for SOS_Spontaneous
times = sos_spontaneous_df_cleaned[sos_spontaneous_df_cleaned["Condition"] == "Spontaneous"]["Time"]
params = scipy.stats.weibull_min.fit(times, floc=0)
print(params)
estimated_mean_spontaneous = estimate_weibull_mean(params=params)
sos_spontaneous_df_cleaned["Time"].astype(int)
sos_spontaneous_df_cleaned.loc[(sos_spontaneous_df_cleaned['Condition'] == 'Spontaneous') & (sos_spontaneous_df_cleaned['Time'] == 86400), 'Log_Time'] = estimated_mean_spontaneous
sos_spontaneous_df_cleaned.info()

(2.342071710113067, 0, 55211.31638074656)
<class 'pandas.core.frame.DataFrame'>
Index: 196 entries, 576 to 771
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    196 non-null    object 
 1   Time         196 non-null    int64  
 2   Dataset      196 non-null    object 
 3   Log_Time     196 non-null    float64
 4   Conjugation  196 non-null    bool   
 5   R_a          196 non-null    bool   
 6   Chemotaxis   196 non-null    bool   
 7   SOS          196 non-null    bool   
dtypes: bool(4), float64(1), int64(1), object(2)
memory usage: 8.4+ KB


In [114]:
# All the 24h limits have been replaced in Log_Time
sos_spontaneous_df_cleaned[sos_spontaneous_df_cleaned['Time']==86400].head()

,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS
681,Spontaneous,86400,SOS_Spontaneous,10.798024,True,True,True,False
695,Spontaneous,86400,SOS_Spontaneous,10.798024,True,True,True,False
699,Spontaneous,86400,SOS_Spontaneous,10.798024,True,True,True,False
704,Spontaneous,86400,SOS_Spontaneous,10.798024,True,True,True,False
708,Spontaneous,86400,SOS_Spontaneous,10.798024,True,True,True,False


### Mixed-Effects Model

In [113]:
# Step 1: Separate Regression Models

# Filter data for Chemotaxis_Diffusion
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()
# Fit OLS regression model for Chemotaxis
model_chemotaxis = sm.OLS.from_formula('Log_Time ~ Chemotaxis', data=chemotaxis_df).fit()
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

# Filter data for Conjugation_Ra
conjugation_ra_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()
# Create the new column with opposite boolean values
conjugation_ra_df["No_R_a"] = ~conjugation_ra_df["R_a"]
# Add interaction term for No_R_a * Conjugation
conjugation_ra_df['No_R_a_Conj'] = conjugation_ra_df['No_R_a'] & conjugation_ra_df['Conjugation']
# Fit OLS regression model for R_a_Conj
model_ra_conj = sm.OLS.from_formula('Log_Time ~ Conjugation + No_R_a_Conj', data=conjugation_ra_df).fit()
model_ra_conj_summary = model_ra_conj.summary()
print(model_ra_conj_summary)



# Filter data for SOS_Spontaneous (LOOK IN ESTIMATING FITTED MEAN SECTION)
sos_spontaneous_df = sos_spontaneous_df_cleaned

# Fit OLS regression model for R_a_Conj
model_sos_spontaneous = sm.OLS.from_formula('Log_Time ~ SOS', data=sos_spontaneous_df).fit()
model_sos_spontaneous_summary = model_sos_spontaneous.summary()
print(model_sos_spontaneous_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.566
Model:                            OLS   Adj. R-squared:                  0.563
Method:                 Least Squares   F-statistic:                     240.8
Date:                Tue, 23 Jul 2024   Prob (F-statistic):           2.50e-35
Time:                        00:11:57   Log-Likelihood:                 216.50
No. Observations:                 187   AIC:                            -429.0
Df Residuals:                     185   BIC:                            -422.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              9.9000      0

In [115]:
# Step 2: Mixed-Effects Model
# Combine both datasets for mixed-effects model
combined_df_mixed = pd.concat([chemotaxis_df, conjugation_ra_df, sos_spontaneous_df], ignore_index=True)
combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis']
# Set 'Chemotaxis' values to NaN for the 'Conjugation_Ra' dataset
combined_df_mixed['Chemotaxis'] = combined_df_mixed['Chemotaxis'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA
combined_df_mixed['No_R_a'] = combined_df_mixed['No_R_a'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA
# Add interaction term for No_R_a * Conjugation
combined_df_mixed['No_R_a_Conj'] = combined_df_mixed['No_R_a'] & combined_df_mixed['Conjugation']


# Verify the changes
combined_df_mixed.info()
combined_df_mixed.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 772 entries, 0 to 771
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    772 non-null    object 
 1   Time         772 non-null    int64  
 2   Dataset      772 non-null    object 
 3   Log_Time     772 non-null    float64
 4   Conjugation  772 non-null    bool   
 5   R_a          772 non-null    bool   
 6   Chemotaxis   772 non-null    bool   
 7   SOS          772 non-null    bool   
 8   No_R_a       772 non-null    bool   
 9   No_R_a_Conj  772 non-null    bool   
dtypes: bool(6), float64(1), int64(1), object(2)
memory usage: 28.8+ KB


,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS,No_R_a,No_R_a_Conj
767,Spontaneous,17196,SOS_Spontaneous,9.752432,True,True,True,False,True,True
768,Spontaneous,46726,SOS_Spontaneous,10.752056,True,True,True,False,True,True
769,Spontaneous,43138,SOS_Spontaneous,10.672160,True,True,True,False,True,True
770,Spontaneous,26322,SOS_Spontaneous,10.178160,True,True,True,False,True,True
771,Spontaneous,27136,SOS_Spontaneous,10.208617,True,True,True,False,True,True


In [116]:
# Prepare the mixed-effects model formula
mixed_formula = 'Log_Time ~ Chemotaxis + Conjugation + No_R_a_Conj + SOS'

# Fit the mixed-effects model
mixed_model = MixedLM.from_formula(mixed_formula, groups='Dataset', data=combined_df_mixed, missing='drop').fit()

# Display the summary of the mixed-effects model
mixed_model_summary = mixed_model.summary()
print(mixed_model_summary)

             Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   Log_Time 
No. Observations:     772       Method:               REML     
No. Groups:           3         Scale:                0.1043   
Min. group size:      187       Log-Likelihood:       -240.5206
Max. group size:      389       Converged:            Yes      
Mean group size:      257.3                                    
---------------------------------------------------------------
                    Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept           10.523    0.571  18.424 0.000  9.403 11.642
Chemotaxis[T.True]  -0.174    0.047  -3.685 0.000 -0.267 -0.081
Conjugation[T.True] -0.171    0.040  -4.240 0.000 -0.251 -0.092
No_R_a_Conj[T.True] -0.211    0.046  -4.554 0.000 -0.302 -0.120
SOS[T.True]         -0.734    0.046 -15.902 0.000 -0.824 -0.643
Dataset Var          0.966    2.996                  